# 02 — Silver | Data Quality & Transformation
**Microsoft Fabric · Lakehouse · PySpark**

Objetivo: transformar a tabela Bronze em dados tipados, consistentes e prontos para análise.

Regras aplicadas:
- Normalização de nomes de colunas.
- Conversão explícita de tipos.
- Remoção de duplicados exatos.
- Tratamento de valores inválidos e nulos sem inventar valores.
- Criação de indicadores de qualidade.
- Separação de registos válidos e rejeitados.

A tabela Silver mantém `loan_status` como variável observada; não é feita previsão de risco nesta etapa.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import re

BRONZE_TABLE = "bronze_credit_risk"
SILVER_TABLE = "silver_credit_risk"
REJECTS_TABLE = "silver_credit_risk_rejects"

In [ ]:
df = spark.table(BRONZE_TABLE)

# Padroniza nomes: minúsculas e underscore
def normalize_col(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

for old in df.columns:
    new = normalize_col(old)
    if old != new:
        df = df.withColumnRenamed(old, new)

display(df.limit(5))
print(df.columns)

In [ ]:
# Conversão explícita dos campos conhecidos do dataset
numeric_int = [
    "person_age", "loan_amnt", "loan_status",
    "cb_person_cred_hist_length"
]
numeric_double = [
    "person_income", "person_emp_length",
    "loan_int_rate", "loan_percent_income"
]

for c in numeric_int:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("int"))

for c in numeric_double:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("double"))

# Normaliza dimensões categóricas
categorical = [
    "person_home_ownership", "loan_intent", "loan_grade",
    "cb_person_default_on_file"
]
for c in categorical:
    if c in df.columns:
        df = df.withColumn(c, F.upper(F.trim(F.col(c))))

In [ ]:
# Deduplicação exata dos atributos de negócio (exclui metadados técnicos)
business_cols = [c for c in df.columns if not c.startswith("_")]
df = df.dropDuplicates(business_cols)

# Regras de validade essenciais
required_cols = [
    "person_age", "person_income", "loan_amnt",
    "loan_status", "loan_intent", "loan_grade"
]
existing_required = [c for c in required_cols if c in df.columns]

invalid_condition = F.lit(False)
for c in existing_required:
    invalid_condition = invalid_condition | F.col(c).isNull()

if "loan_status" in df.columns:
    invalid_condition = invalid_condition | (~F.col("loan_status").isin(0, 1))
if "person_age" in df.columns:
    invalid_condition = invalid_condition | (F.col("person_age") <= 0)
if "person_income" in df.columns:
    invalid_condition = invalid_condition | (F.col("person_income") <= 0)
if "loan_amnt" in df.columns:
    invalid_condition = invalid_condition | (F.col("loan_amnt") <= 0)
if "loan_int_rate" in df.columns:
    invalid_condition = invalid_condition | (F.col("loan_int_rate") < 0)
if "loan_percent_income" in df.columns:
    invalid_condition = invalid_condition | (F.col("loan_percent_income") < 0)

df_checked = df.withColumn("_is_valid", ~invalid_condition)
df_valid = (
    df_checked.filter(F.col("_is_valid"))
    .drop("_is_valid")
    .withColumn("_silver_processed_at", F.current_timestamp())
)
df_rejected = (
    df_checked.filter(~F.col("_is_valid"))
    .withColumn("_rejection_reason", F.lit("Null em campo obrigatório ou valor fora das regras básicas"))
    .withColumn("_silver_processed_at", F.current_timestamp())
)

print("Válidos:", df_valid.count())
print("Rejeitados:", df_rejected.count())

In [ ]:
# Persiste Silver e tabela de rejeitados para auditoria
(
    df_valid.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

(
    df_rejected.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(REJECTS_TABLE)
)

display(spark.table(SILVER_TABLE).limit(10))
display(spark.table(REJECTS_TABLE).limit(10))

In [ ]:
# Perfil de qualidade: nulos por coluna
silver = spark.table(SILVER_TABLE)
null_counts = silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver.columns if not c.startswith("_")
])
display(null_counts)

print("Total Silver:", silver.count())
print("Total rejeitados:", spark.table(REJECTS_TABLE).count())

## Resultado esperado
- `silver_credit_risk`: registos validados e tipados.
- `silver_credit_risk_rejects`: registos que falharam regras essenciais.
- Perfil simples de nulos para documentar qualidade.

**Importante:** as regras são deliberadamente conservadoras. Antes de uma utilização real, analisar os rejeitados e validar limites de negócio com especialistas. Não imputar rendimento, taxa ou histórico de crédito sem uma regra documentada.